# 03. Build the database
Clean the raw data, convert everything to hourly values, add recent OMIE prices, and save it all in a SQLite database.

In [14]:
# Import the libraries
import sqlite3
import requests
import pandas as pd

In [15]:
# Load the REN prices and use hours from 0 to 23 (REN uses 1 to 24)
prices = pd.read_csv("../data/raw/ren_prices_hourly.csv")
prices["hour"] = prices["hour"] - 1

# Keep normal hours only (clock change days in October have a 25th hour)
prices = prices[prices["hour"] <= 23]
prices["source"] = "REN"
prices = prices[["date", "hour", "price_pt", "price_es", "source"]]
print(len(prices), "hourly prices from REN")

65680 hourly prices from REN


In [16]:
# Download one day of OMIE prices and turn it into a table (one row per 15-minute period)
def read_omie_day(day):
    url = "https://www.omie.es/en/file-download?parents%5B0%5D=marginalpdbc&filename=marginalpdbc_" + day + ".1"
    response = requests.get(url)
    rows = []
    for line in response.text.splitlines():
        parts = line.split(";")
        if len(parts) >= 6 and parts[0].isdigit():
            rows.append({
                "date": parts[0] + "-" + parts[1] + "-" + parts[2],
                "period": int(parts[3]),
                "price_pt": float(parts[4]),
                "price_es": float(parts[5]),
            })
    return pd.DataFrame(rows)

# Download OMIE prices from 30 June 2026 until tomorrow
omie_days = []
for day in pd.date_range("2026-06-30", "2026-10-03"):
    omie_days.append(read_omie_day(day.strftime("%Y%m%d")))
omie = pd.concat(omie_days)

# Turn the 15-minute periods into hourly averages
omie["hour"] = (omie["period"] - 1) // 4
omie = omie.groupby(["date", "hour"])[["price_pt", "price_es"]].mean().reset_index()
omie["source"] = "OMIE"
print(len(omie), "hourly prices from OMIE")

2304 hourly prices from OMIE


In [17]:
# Put the REN and OMIE prices together, sorted by date and hour
all_prices = pd.concat([prices, omie])
all_prices = all_prices.sort_values(["date", "hour"])
print(len(all_prices), "hourly prices in total, from", all_prices["date"].min(), "to", all_prices["date"].max())

67984 hourly prices in total, from 2019-01-01 to 2026-10-03


In [18]:
# Prices are in Spanish time (CET), one hour ahead of Portugal: move them back one hour
all_prices["datetime"] = pd.to_datetime(all_prices["date"]) + pd.to_timedelta(all_prices["hour"], unit="h") - pd.Timedelta(hours=1)
all_prices["date"] = all_prices["datetime"].dt.strftime("%Y-%m-%d")
all_prices["hour"] = all_prices["datetime"].dt.hour
all_prices = all_prices.drop(columns=["datetime"])

# Keep data from 2019 only (the first hour moved to 31 December 2018)
all_prices = all_prices[all_prices["date"] >= "2019-01-01"]
print(len(all_prices), "hourly prices, now in Portuguese time")

67983 hourly prices, now in Portuguese time


In [19]:
# Load the REN production and turn the 15-minute values into hourly averages
production = pd.read_csv("../data/raw/ren_production_15min.csv")
production["hour"] = production["time"].str[:2].astype(int)
production = production.drop(columns=["time"])
production = production.groupby(["date", "hour"]).mean().reset_index()

# Simple column names for SQL: lower case and no spaces (e.g. "Natural Gas" becomes "natural_gas")
new_names = {}
for col in production.columns:
    new_names[col] = col.lower().replace(" + ", "_plus_").replace(" ", "_")
production = production.rename(columns=new_names)

print(len(production), "hourly production rows")
production.head()

67936 hourly production rows


,date,hour,consumption_plus_storage,consumption,pumping,hydro,solar,import,export,wind,natural_gas,other_thermal,biomass,coal,wave,consumption_of_batteries,battery_injection
0,2019-01-01,0,4796.900,4796.900,0.000,1283.275,0.0,740.575,0.0,542.575,433.475,19.025,322.825,1471.9,NaN,NaN,NaN
1,2019-01-01,1,4649.600,4649.600,0.000,1038.175,0.0,949.550,0.0,440.475,427.350,17.925,323.225,1469.6,NaN,NaN,NaN
2,2019-01-01,2,4699.950,4394.400,305.550,715.125,0.0,1383.150,0.0,366.850,427.575,21.250,334.275,1469.8,NaN,NaN,NaN
3,2019-01-01,3,4854.475,4176.600,677.875,335.050,0.0,1988.175,0.0,346.225,429.400,25.100,342.650,1406.6,NaN,NaN,NaN
4,2019-01-01,4,4715.325,4016.975,698.350,244.200,0.0,1994.750,0.0,413.050,429.625,26.250,339.125,1289.1,NaN,NaN,NaN


In [20]:
# Save both tables in a SQLite database file
connection = sqlite3.connect("../data/electricity.db")
all_prices.to_sql("prices", connection, if_exists="replace", index=False)
production.to_sql("production", connection, if_exists="replace", index=False)
print("Database saved")

Database saved


## First SQL queries

In [21]:
# Average price in Portugal by year
query = """
SELECT substr(date, 1, 4) AS year,
       ROUND(AVG(price_pt), 1) AS avg_price
FROM prices
GROUP BY year
ORDER BY year
"""
pd.read_sql(query, connection)

,year,avg_price
0,2019,47.9
1,2020,34.0
2,2021,112.0
3,2022,167.9
4,2023,88.3
5,2024,63.5
6,2025,66.2
7,2026,75.1


In [22]:
# The 5 cheapest hours of the day on average in 2025
query = """
SELECT hour,
       ROUND(AVG(price_pt), 1) AS avg_price
FROM prices
WHERE date LIKE '2025%'
GROUP BY hour
ORDER BY avg_price
LIMIT 5
"""
pd.read_sql(query, connection)

,hour,avg_price
0,13,29.7
1,12,30.9
2,14,32.0
3,11,33.4
4,10,37.1


In [23]:
# Average solar and wind production by year (MW)
query = """
SELECT substr(date, 1, 4) AS year,
       ROUND(AVG(solar), 0) AS avg_solar,
       ROUND(AVG(wind), 0) AS avg_wind
FROM production
GROUP BY year
ORDER BY year
"""
pd.read_sql(query, connection)

,year,avg_solar,avg_wind
0,2019,121.0,1532.0
1,2020,145.0,1373.0
2,2021,196.0,1474.0
3,2022,289.0,1477.0
4,2023,410.0,1475.0
5,2024,559.0,1610.0
6,2025,697.0,1539.0
7,2026,840.0,1431.0


In [24]:
# Average solar production by hour in June 2025 (should peak around 13:00 to 14:00 Portuguese summer time)
query = """
SELECT hour,
       ROUND(AVG(solar), 0) AS avg_solar
FROM production
WHERE date LIKE '2025-06%'
GROUP BY hour
ORDER BY hour
"""
pd.read_sql(query, connection)

,hour,avg_solar
0,0,0.0
1,1,0.0
2,2,0.0
3,3,0.0
4,4,0.0
5,5,0.0
6,6,56.0
7,7,425.0
8,8,1072.0
9,9,1649.0


In [25]:
# Average price by hour in June 2025
query = """
SELECT hour,
       ROUND(AVG(price_pt), 1) AS avg_price
FROM prices
WHERE date LIKE '2025-06%'
GROUP BY hour
ORDER BY hour
"""
pd.read_sql(query, connection)

,hour,avg_price
0,0,100.8
1,1,97.6
2,2,95.9
3,3,95.7
4,4,97.1
5,5,102.3
6,6,103.0
7,7,86.6
8,8,58.8
9,9,41.6


In [26]:
   # Save both tables as CSV files (these go to GitHub and are updated every day)
   all_prices.round(2).to_csv("../data/prices_hourly.csv", index=False, lineterminator="\n")
   production.round(1).to_csv("../data/production_hourly.csv", index=False, lineterminator="\n")